# SARIMA on Latest R1 Features Dataset

This notebook builds a SARIMA workflow on top of the latest `features_YYYYMMDD_HHMMSS.csv` file in `data/processed/`.

The goal is not to produce a final production model yet, but to create a transparent baseline that lets you study:

- hourly seasonality in delay
- differences across `day_type`
- whether a simple SARIMA model can capture the trend/seasonality you observed in the EDA

The notebook works on the processed dataset after filtering out rows with missing `direction`, because that is the working subset used in the main EDA notebook.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)

In [ ]:
# Resolve paths relative to this notebook.
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"


def latest_features_file(processed_dir: Path) -> Path:
    candidates = sorted(
        path
        for path in processed_dir.glob("features_*.csv")
        if re.fullmatch(r"features_\d{8}_\d{6}\.csv", path.name)
    )
    if not candidates:
        raise FileNotFoundError("No timestamped features CSV found in data/processed/.")
    return candidates[-1]


def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

In [ ]:
# Load the latest features file and parse timestamps.
features_path = latest_features_file(PROCESSED_DIR)
df = pd.read_csv(features_path)

for col in ["planned_arrival_dt", "actual_arrival_dt", "hour_trunc", "service_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df_model = df[df["direction"].notna()].copy()

print(f"Loaded features file: {features_path.name}")
print(f"Original rows: {len(df):,}")
print(f"Rows with non-missing direction: {len(df_model):,}")
print(f"Date range: {df_model['service_date'].min().date()} -> {df_model['service_date'].max().date()}")
print("\nDay type counts (rows):")
display(df_model["day_type"].value_counts())

### Interpretation

On the current latest file, the filtered modeling table keeps `5747` rows out of `6090`, so the missing-direction removal is relatively small.

The day-type mix is currently dominated by `weekend` and `workday`, while `holiday` is very small and `holiday&weekend` is absent in this sample. That means any model-based conclusions will be much more reliable for workdays and weekends than for holidays.

## 1. Build Hourly Time Series

SARIMA expects a univariate time series indexed at a regular frequency.

Here we aggregate the processed train-station rows into an **hourly average delay series**. We do this separately by `day_type`, because your EDA suggested different seasonality patterns for workdays and weekends.

The main target is hourly mean `target_delay`.

In [ ]:
def build_hourly_delay_series(frame: pd.DataFrame, day_type: str | None = None) -> pd.Series:
    subset = frame.copy()
    if day_type is not None:
        subset = subset[subset["day_type"] == day_type].copy()

    subset = subset.dropna(subset=["hour_trunc", "target_delay"])
    hourly = (
        subset.groupby("hour_trunc")["target_delay"]
        .mean()
        .sort_index()
    )

    full_index = pd.date_range(hourly.index.min(), hourly.index.max(), freq="H")
    hourly = hourly.reindex(full_index)
    hourly = hourly.interpolate(method="time").ffill().bfill()
    hourly.index.name = "timestamp"
    return hourly


series_workday = build_hourly_delay_series(df_model, "workday")
series_weekend = build_hourly_delay_series(df_model, "weekend")
series_holiday = build_hourly_delay_series(df_model, "holiday") if (df_model['day_type'] == 'holiday').any() else None
series_holiday_weekend = build_hourly_delay_series(df_model, "holiday&weekend") if (df_model['day_type'] == 'holiday&weekend').any() else None

series_summary = pd.DataFrame({
    "day_type": ["workday", "weekend", "holiday", "holiday&weekend"],
    "n_hours": [
        len(series_workday),
        len(series_weekend),
        len(series_holiday) if series_holiday is not None else 0,
        len(series_holiday_weekend) if series_holiday_weekend is not None else 0,
    ],
})
display(series_summary)

## 2. Visualize the Hourly Series

Before fitting SARIMA, it is useful to inspect the aggregated series directly. This lets you verify whether the workday/weekend seasonality you saw in the EDA is still visible after aggregation to an hourly index.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False)
axes = axes.flatten()

series_map = {
    'workday': series_workday,
    'weekend': series_weekend,
    'holiday': series_holiday,
    'holiday&weekend': series_holiday_weekend,
}

for ax, (label, series) in zip(axes, series_map.items()):
    if series is None or len(series) == 0:
        ax.set_visible(False)
        continue
    ax.plot(series.index, series.values, color="#4C72B0")
    ax.set_title(label)
    ax.set_ylabel("hourly mean target_delay")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Hourly Delay Series by Day Type", y=1.02)
plt.tight_layout()
plt.show()

## 3. Mean Hour-of-Day Seasonality

This reproduces the seasonality pattern in a way that is useful for model design: average delay by hour of day, split by `day_type`.

In [ ]:
seasonality = (
    df_model.groupby(["day_type", "hour"])['target_delay']
    .mean()
    .reset_index(name='avg_target_delay')
)

day_type_order = ["workday", "weekend", "holiday", "holiday&weekend"]
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, day_type in zip(axes, day_type_order):
    subset = seasonality[seasonality["day_type"] == day_type]
    if subset.empty:
        ax.set_visible(False)
        continue
    sns.lineplot(data=subset, x="hour", y="avg_target_delay", marker="o", ax=ax, color="#55A868")
    ax.set_title(day_type)
    ax.set_xlabel("hour of day")
    ax.set_ylabel("average target_delay")

plt.suptitle("Average Delay by Hour and Day Type", y=1.02)
plt.tight_layout()
plt.show()

### Interpretation

The current seasonality pattern is noticeably stronger on `workday` than on `weekend`.

For the latest file, the workday hourly average delay peaks around hour `13` at roughly `18.9` minutes and is lowest around hour `0` at roughly `0.9` minutes. Weekend seasonality is milder, with a peak around hour `12` at roughly `6.2` minutes.

The `holiday` panel should be treated very cautiously because it is based on only a small number of dates, so its shape is much noisier and less trustworthy.

## 4. ACF and PACF Diagnostics

For an hourly series, a natural seasonal period to try is `24`, corresponding to daily repetition. The ACF/PACF plots help you sanity-check whether that assumption is plausible.

We focus on workday first because it is usually the richest subset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(series_workday, lags=72, ax=axes[0])
axes[0].set_title("ACF - Workday Hourly Delay")
plot_pacf(series_workday, lags=72, ax=axes[1], method="ywm")
axes[1].set_title("PACF - Workday Hourly Delay")
plt.tight_layout()
plt.show()

## 5. Train/Test Split

We keep the last few days as a holdout set. For an hourly seasonal model, a holdout of 72 hours (3 days) is a reasonable starting point.

In [ ]:
FORECAST_HORIZON = 72
SEASONAL_PERIOD = 24


def train_test_split_series(series: pd.Series, horizon: int = 72):
    train = series.iloc[:-horizon].copy()
    test = series.iloc[-horizon:].copy()
    return train, test


workday_train, workday_test = train_test_split_series(series_workday, FORECAST_HORIZON)
weekend_train, weekend_test = train_test_split_series(series_weekend, FORECAST_HORIZON)

print(f"Workday train size: {len(workday_train)} | test size: {len(workday_test)}")
print(f"Weekend train size: {len(weekend_train)} | test size: {len(weekend_test)}")

## 6. Fit Baseline SARIMA Models

We start with a conservative baseline specification:

- non-seasonal order: `(1, 0, 1)`
- seasonal order: `(1, 0, 1, 24)`

This is not guaranteed to be optimal, but it is a reasonable first model when the EDA suggests a daily hourly pattern.

In [ ]:
def fit_sarima(series: pd.Series, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24)):
    model = SARIMAX(
        series,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    result = model.fit(disp=False)
    return result


sarima_workday = fit_sarima(workday_train, order=(1, 0, 1), seasonal_order=(1, 0, 1, SEASONAL_PERIOD))
sarima_weekend = fit_sarima(weekend_train, order=(1, 0, 1), seasonal_order=(1, 0, 1, SEASONAL_PERIOD))

print(sarima_workday.summary())

## 7. Forecast on Holdout

We now forecast the holdout period and compare predictions against the observed hourly mean delay series.

In [ ]:
def forecast_and_evaluate(result, train: pd.Series, test: pd.Series, label: str):
    forecast = result.get_forecast(steps=len(test))
    pred_mean = forecast.predicted_mean
    conf_int = forecast.conf_int()

    metrics = pd.DataFrame({
        "series": [label],
        "mae": [mae(test, pred_mean)],
        "rmse": [rmse(test, pred_mean)],
    })

    return pred_mean, conf_int, metrics


workday_pred, workday_ci, workday_metrics = forecast_and_evaluate(sarima_workday, workday_train, workday_test, "workday")
weekend_pred, weekend_ci, weekend_metrics = forecast_and_evaluate(sarima_weekend, weekend_train, weekend_test, "weekend")

display(pd.concat([workday_metrics, weekend_metrics], ignore_index=True))

### Interpretation

For the current latest dataset, the baseline SARIMA behaves much better on `workday` than on `weekend`.

Current holdout metrics are approximately:

- `workday`: `MAE ≈ 2.09`, `RMSE ≈ 2.29`
- `weekend`: `MAE ≈ 4.02`, `RMSE ≈ 7.71`

This suggests that workday delay dynamics are more regular and easier for a daily seasonal model to capture. The weekend series appears more volatile, with larger forecast errors and likely stronger outliers or irregular events.

So at this stage, SARIMA looks like a more credible baseline for workdays than for weekends.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

axes[0].plot(workday_train.index[-7*24:], workday_train.iloc[-7*24:], label="train tail", color="#4C72B0")
axes[0].plot(workday_test.index, workday_test, label="actual test", color="#C44E52")
axes[0].plot(workday_pred.index, workday_pred, label="forecast", color="#55A868")
axes[0].fill_between(workday_ci.index, workday_ci.iloc[:, 0], workday_ci.iloc[:, 1], color="#55A868", alpha=0.2)
axes[0].set_title("SARIMA Forecast - Workday")
axes[0].legend()

axes[1].plot(weekend_train.index[-7*24:], weekend_train.iloc[-7*24:], label="train tail", color="#4C72B0")
axes[1].plot(weekend_test.index, weekend_test, label="actual test", color="#C44E52")
axes[1].plot(weekend_pred.index, weekend_pred, label="forecast", color="#55A868")
axes[1].fill_between(weekend_ci.index, weekend_ci.iloc[:, 0], weekend_ci.iloc[:, 1], color="#55A868", alpha=0.2)
axes[1].set_title("SARIMA Forecast - Weekend")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Residual Diagnostics

Residual diagnostics help you judge whether the model has captured most of the structure or whether strong autocorrelation remains.

In [ ]:
workday_resid = sarima_workday.resid.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(workday_resid, bins=40, kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Workday Residual Distribution")
plot_acf(workday_resid, lags=72, ax=axes[1])
axes[1].set_title("ACF of Workday Residuals")
plt.tight_layout()
plt.show()

## 9. Optional Extensions

Natural next steps after this baseline:

- tune `(p, d, q)` and `(P, D, Q, 24)` more systematically
- fit separate models for `holiday` and `holiday&weekend` if there are enough observations
- add exogenous regressors via `SARIMAX`, for example:
  - `is_holiday`
  - `is_weekend`
  - weather variables if merged later
  - station-level or direction-level aggregates
- compare hourly mean delay against other targets, for example hourly median delay
- compare SARIMA against simpler baselines such as seasonal naive forecasts